<h1>Chapters 5 - Skills</h1>
<i>Adding specialized skills to your Agent.</i>


<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents/blob/main/chapter05/chapter05_skills.ipynb)

---

This notebook is for Chapter 5 of [An Illustrated Guide to AI Agents](https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ) by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use. The skills that we are going to explore can be activated as if they are tools. As such, you can either use the non-native (Gemma 3) or native (Gemma 4) since both have tool calling capabilities that work with skills. For simplicity, we are going with Gemma 4 since we continue to focus more on native capabilities from this point onward:

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B (with native thinking and tool calling)
llm = LLM(model="gemma4:e4b")

If you want to use another LLM, here are a couple of options (both locally and on the cloud) that you can try:

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-4-e4B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-4-e4b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-4-31b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2 - Adding Recipes with **`SKILL.md`**

With modules like tools and MCP, we can give an Agent access to a number of tools or actions that it can take. However, when exactly to use those actions and how they fit into a larger workflow is not covered by any of these modules. This is where `SKILL.md` come in, they are a set of instructions on how to perform a given task. For instance, if you want it to create a presentation, you might need it to first search the web for relevant information (this is a tool), then summarize this information (it can do this by itself), and then finally create the slides one at a time (this is also a tool). This workflow or "recipe" for creating a presentation might therefore include a set of tools but also instructions on how to use them and in what order. As such,

> Skills teach your Agent what to do, when to do it, and how

The format of a `SKILL.md` file allows for proper context engineering. In the yaml frontmatter, there is the basic description of your skill which is always loaded into the context window:

```yaml
name: ...
description: ...
```

Below that, there is a more extensive description of the skill and how it should be executed. The full structure then becomes something like this:


```markdown
---
name: ...
description ...
---

#
...

##
...
```

This structure is especially helpful as you can write down extensive descriptions on how to use the skill, which may include domain-specific information. Skills are therefore especially helpful when you notice you have to repeat prompts often, like having to explain everytime the tone of voice that you want to or some domain-specific information regarding the schemas of your database.

Another benefit of skills is that they are **progressively disclosed**. This means that the yaml frontmatter is always given to the Agent as a system prompt, much like the tools we constructed in Chapter 5. However, the full markdown description is only given when the skill is **activated**. This therefore occupies a small amount of the context window and extends only when the skill is activated.

Next, let's explore how to give your `TinyAgent` access to these skills by first creating the `Skills` module. This module actually inherits from `Tools`. Why? Because for an Agent, everything is a tool, that's why!

In [ ]:
import inspect
import yaml
from pathlib import Path
from illustrated_agents.chapters.ch5 import NativeTools

class Skills(NativeTools):
    """A `Tools` and `Skills` registry

    Skills are recipes. When you activate one, you get instructions
    in return on how to approach a given task. Although it is not
    a tool in the same way a calculator is one, we can still approach
    it as such since the Agent has to decide when to activate it.

    The skills are loaded progressively. As such, the name and description are
    available in the system prompt, but the full instructions are only injected
    when the agent **activates** a skill (uses it as a tool).
    """

    def add_skill(self, path: str):
        """Load a SKILL.md from a skill folder and register it as a tool."""
        skill_dir = Path(path)
        content = (skill_dir / "SKILL.md").read_text(encoding="utf-8")
        
        # TIER 2: Instructions - Full instructions of the main SKILL.md
        _, frontmatter, instructions = content.split("---", 2)
        meta = yaml.safe_load(frontmatter)

        # TIER 1: Catalog - Name and description of a skill
        name, description = meta["name"], meta["description"]
        
        # TIER 3: List files in the same folder that the agent can access
        extra_files = [p.name for p in skill_dir.iterdir() if p.name != "SKILL.md"]
        
        # When activated, return the instructions plus the folder context
        def skill(**kwargs):
            return f"""{instructions}

Skill folder: {skill_dir}

Available files: {extra_files}"""
        
        skill.__name__ = name
        skill.__doc__ = "A skill. " + description
        skill.__signature__ = inspect.Signature()
        self.add_tool(name, skill, "A skill. " + description)

    @property
    def prompt(self):
        return """You have specialized skills available.
To activate a skill, call it with an empty argument object.
Skills return instructions; they do not process the task directly."""

Much like with the `Tools` module, there are only a couple of functions that we really need to add skills, starting with the prompt:

Now that you have explored the code for loading and activating skills, let's explore how to actually create a skill. We already have prepared a skill for you to use, which can be found in `src/illustrated_agents/skills/file_analyzer`. The file contains all information about how to use our previously defined tool (`read_markdown`) along with a set of instructions on how to summarize its content. Let's load the skill and inspect it:

In [ ]:
import illustrated_agents

# We choose the file_analyzer skill as an example
file_analyzer_path = Path(illustrated_agents.__file__).parent / "skills" / "file_analyzer"

# Load the skill
tools_and_skills = Skills()
tools_and_skills.add_skill(file_analyzer_path)

Let's inspect the skill's description:

In [ ]:
tools_and_skills.registry["file_analyzer"]["description"]

This is a short description of what the task is, but how to actually do it is covered in the full instruction which we can only see when the skill is "executed":

In [ ]:
tools_and_skills.prompt

In [ ]:
from illustrated_agents.chapters.ch2 import Response

response = Response(
  content="",
  tool_call={"function": {"name": "file_analyzer", "arguments": {}}},
)

response = tools_and_skills.parse(response)
print(tools_and_skills.execute(response))

As you can see, the instruction is quite long and adding that to the system prompt would quickly fill up the context window if you have several skills that your Agent can use. 

## 3 - Running your **`TinyAgent`**

Since we approach skills as tools, there is no need to update your `TinyAgent`:

In [ ]:
import requests
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import TinyAgent

def read_markdown(path: str) -> str:
    """Read the content of a markdown file.

    Args:
        path (str): The path to the markdown file.
    """
    return requests.get(path).text

# Tools and Skills
tools_and_skills = Skills()
tools_and_skills.add_skill(file_analyzer_path)
tools_and_skills.add_tool("read_markdown", read_markdown)

# Create agent
agent = TinyAgent(
    llm=llm,
    tools=tools_and_skills,
    memory=Memory(),
)

In [ ]:
query = "Use the `file_analyzer` skill to summarize https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/refs/heads/main/README.md."

print(agent.run(query))

Note that your `TinyAgent` correctly uses the tool and get's back the content of the SKILL.md. What you see above is the observation of calling the `file_analyzer` skill!

The next step would be for the `TinyAgent` to continue and use that the observation (the content of the skill) to complete the task. However, it can only call a tool once and does not yet have autonomous capabilities! That's left for the next chapter. 

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered how to give your `TinyAgent` skills which allows for it to get additional context and proper instructions on how to approach certain procedures. 

In [ ]:
from illustrated_agents.chapters.ch5 import what_we_built_skills; what_we_built_skills

# What's Next

In the next chapter, we will finally introduce the one powerful mechanism that converts the LLM into an Agent... the **for loop**! (Reason and Act to be specific 😉)